<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/main/phase_01_data_exploration/data_exploration_of_sub_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading and Extracting COCO 2017 Dataset
This notebook downloads the annotations, training dataset, and validation dataset from COCO 2017, and then extracts them.

# Download The data set + Unzip it.

In [ ]:
# 1. Download annotations, train images, and validation images
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/train2017.zip
!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip

annotations_trainva 100%[===================>] 241.19M  20.9MB/s    in 13s     
train2017.zip       100%[===================>]  18.01G  20.5MB/s    in 15m 23s 
val2017.zip         100%[===================>] 777.80M  21.3MB/s    in 38s     


In [ ]:
# 2. Extract all downloaded zip files
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip

Extracting annotations...


In [ ]:
print("Extracting training dataset (this may take a while)...")
!unzip -q train2017.zip

Extracting training dataset (this may take a while)...


In [ ]:
print("Extracting validation dataset...")
!unzip -q val2017.zip

Extracting validation dataset...


# Data Exploration
Calculation of metrics

## Distribution of object classes

In [9]:
from collections import Counter
import json
import os

# Google Colab path definitions
INSTANCES_TRAIN_FILE = os.path.join("annotations", "instances_train2017.json")
INSTANCES_VAL_FILE = os.path.join("annotations", "instances_val2017.json")

TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def build_image_to_categories_map(instances_data):
    """Maps image_id -> list of object category IDs present in that image."""
    image_to_categories = {}
    for ann in instances_data["annotations"]:
        image_id = ann["image_id"]
        cat_id = ann["category_id"]
        image_to_categories.setdefault(image_id, []).append(cat_id)
    return image_to_categories


def count_top_classes_for_split(
    split_data, image_to_categories_map, category_name_map, top_k=50
):
    """Calculates object instance frequencies for a split and returns Top K."""
    class_counts = Counter()
    for item in split_data:
        image_id = item["image_id"]
        # Extract the catogories existing in the current image
        cat_ids = image_to_categories_map.get(image_id, [])
        for cid in cat_ids:
            # Get category name
            cat_name = category_name_map.get(cid, f"Unknown_{cid}")
            # Increase category count
            class_counts[cat_name] += 1
    # order (descend) and return top k
    return class_counts.most_common(top_k)


def display_top_classes(split_name, top_classes):
    """Prints a formatted table of Top 50 classes and their occurrence counts."""
    print(f"\n==================================================")
    print(f"   Top 50 Object Classes: {split_name}")
    print(f"==================================================")
    print(f"{'Rank':<6} | {'Category Name':<22} | {'Occurrences':<12}")
    print("-" * 48)
    for rank, (cat_name, count) in enumerate(top_classes, start=1):
        print(f"{rank:<6} | {cat_name:<22} | {count:<12}")


def run_object_distribution_analysis():
    print("Loading COCO instance annotations...")
    train_instances = read_json(INSTANCES_TRAIN_FILE)
    val_instances = read_json(INSTANCES_VAL_FILE)

    # Build category ID to Name map
    category_name_map = {
        cat["id"]: cat["name"] for cat in train_instances["categories"]
    }

    # Build image_id -> category_ids lookup maps
    train_image_to_cats = build_image_to_categories_map(train_instances)
    val_image_to_cats = build_image_to_categories_map(val_instances)

    # 1. Process Test Split (sourced from val2017)
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        top_test = count_top_classes_for_split(
            test_data, val_image_to_cats, category_name_map
        )
        display_top_classes("Test Subset (test_100.json)", top_test)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 2. Process Validation Split (sourced from train2017)
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        top_val = count_top_classes_for_split(
            val_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Validation Subset (val_100.json)", top_val)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split (sourced from train2017)
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        top_kb = count_top_classes_for_split(
            kb_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Knowledge Base Subset (kb_10k.json)", top_kb)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_object_distribution_analysis()

Loading COCO instance annotations...

   Top 50 Object Classes: Test Subset (test_100.json)
Rank   | Category Name          | Occurrences 
------------------------------------------------
1      | person                 | 85          
2      | vase                   | 41          
3      | bottle                 | 30          
4      | toilet                 | 29          
5      | book                   | 28          
6      | clock                  | 17          
7      | zebra                  | 17          
8      | bus                    | 16          
9      | refrigerator           | 11          
10     | teddy bear             | 10          
11     | cat                    | 9           
12     | giraffe                | 8           
13     | potted plant           | 8           
14     | remote                 | 8           
15     | sheep                  | 7           
16     | truck                  | 7           
17     | couch                  | 7           
18     | car 